In [ ]:
import datetime
import os

import pandas as pd

from modules.predictor.data.utils import prepare_data_for_regressors, custom_data_kfold, combine_split, \
    complex_data_conversion
from modules.predictor.training_and_evaluation.train_eval_pipeline import batch_train_and_eval

In [ ]:
num_splits = 4
num_bins = 4
random_state = 42

dataset = 'dft'

if dataset == 'custom':
    features_all = ['flatness', 'PS', '%C', '%N', '%O', 'MolLogP', 'num_atoms', 'num_bonds', 'num_aromatic_rings',
                    'num_heteroatoms', 'num_rotatable_bonds', 'num_h_acceptors', 'num_h_donors', 'tpsa', 'mol_wt',
                    'symmetry_C2', 'symmetry_C2h', 'symmetry_C2v', 'symmetry_Cs', 'symmetry_D2', 'symmetry_D2h']
else:
    features_all = ['total_energy', 'homo', 'lumo', 'dipole_x', 'diploe_y', 'dipole_z', 'vibrational_frequencies_mean', 'vibrational_frequencies_max', 'vibrational_frequencies_min', 'internal_energy_0K', 'internal_energy_298K', 'zpe', 'free_enegry', 'enthalpy_energy']
target = 'capacity_max'

In [ ]:
experts1_name = 'data_experts1.csv'
experts2_name = 'data_experts2.csv'
saad_name = 'data_saad.csv'
zhu_name = 'data_zhu.csv'

path_custom = '../../../data/processed_selected_custom_features'
path_dft = '../../../data/processed_dft_features'

if dataset == 'custom':
    experts1_path = os.path.join(path_custom, experts1_name)
    experts2_path = os.path.join(path_custom, experts2_name)
    saad_path = os.path.join(path_custom, saad_name)
    zhu_path = os.path.join(path_custom, zhu_name)
else:
    experts1_path = os.path.join(path_dft, experts1_name)
    experts2_path = os.path.join(path_dft, experts2_name)
    saad_path = os.path.join(path_dft, saad_name)
    zhu_path = os.path.join(path_dft, zhu_name)

# Model training and eval - only experts1 data

## Data preparation and split

In [ ]:
df_experts1 = pd.read_csv(experts1_path)

df_experts1.head()

In [ ]:
if dataset == 'dft':
    features_fix = ['vibrational_frequencies_mean', 'vibrational_frequencies_max', 'vibrational_frequencies_min']
    df_experts1, new_features = complex_data_conversion(df_experts1, features_fix)
    features_all = [f for f in features_all if f not in features_fix] + new_features
    df_experts1 = df_experts1[features_all + [target, 'smiles']]
df_experts1.head()

In [ ]:
folds_experts1 = custom_data_kfold(df_experts1, target, num_splits, num_bins, random_state)

for (train, test) in folds_experts1:
    print(len(train), len(test))

In [ ]:
df_experts1.drop(columns=['smiles'], inplace=True)

if dataset == 'custom':
    cat_features = ['symmetry']
    num_features = [f for f in df_experts1.columns if f not in [target]]
else:
    cat_features = []
    num_features = [f for f in df_experts1.columns if f not in [target]]

df_experts1 = prepare_data_for_regressors(df_experts1, num_features, cat_features)
features = [f for f in df_experts1 if f in features_all]
df_experts1[features].head()

## Model evaluation

In [ ]:
date = datetime.date.today().strftime('%d-%m-%Y')
save_dir = f'../../../results_{dataset}/expert1/{date}/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
batch_train_and_eval(df_experts1, 'expert1', folds_experts1, target, features, 'grid_search', save_dir)

# Model training and eval - only expert data (experts1 and experts2)

## Data preparation and split

In [ ]:
df_experts1 = pd.read_csv(experts1_path)
df_experts2 = pd.read_csv(experts2_path)

In [ ]:
folds_experts1 = custom_data_kfold(df_experts1, target, num_splits, num_bins, random_state)
folds_experts2 = custom_data_kfold(df_experts2, target, num_splits, num_bins, random_state)

folds_experts, df_experts = combine_split(df_experts1, folds_experts1, df_experts2, folds_experts2)

print(len(df_experts))
for (train, test) in folds_experts:
    print(len(train), len(test))

In [ ]:
if dataset == 'dft':
    features_fix = ['vibrational_frequencies_mean', 'vibrational_frequencies_max', 'vibrational_frequencies_min']
    df_experts, new_features = complex_data_conversion(df_experts, features_fix)
    features_all = [f for f in features_all if f not in features_fix] + new_features
    df_all = df_experts[features_all + [target, 'smiles']]
df_experts.head()

In [ ]:
df_experts.drop(columns=['smiles'], inplace=True)

if dataset == 'custom':
    cat_features = ['symmetry']
    num_features = [f for f in df_experts.columns if f not in [target]]
elif dataset == 'dft':
    cat_features = []
    num_features = [f for f in df_experts.columns if f not in [target]]

df_experts = prepare_data_for_regressors(df_experts, num_features, cat_features)
features = [f for f in df_experts if f in features_all]
df_experts.head()

## Model evaluation

In [ ]:
date = datetime.date.today().strftime('%d-%m-%Y')
save_dir = f'../../../results_{dataset}/expert/{date}/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
batch_train_and_eval(df_experts, 'expert', folds_experts, target, features, 'grid_search', save_dir)

# Model training and eval - all datasets (experts1, experts2, zhu, saad)

## Data preparation and split

In [ ]:
df_experts1 = pd.read_csv(experts1_path)
df_experts2 = pd.read_csv(experts2_path)
df_zhu = pd.read_csv(zhu_path)
df_saad = pd.read_csv(saad_path)

df_rest = pd.concat([df_zhu, df_saad, df_experts2], ignore_index=True)
print(len(df_rest))

In [ ]:
folds_experts1 = custom_data_kfold(df_experts1, target, num_splits, num_bins, random_state)
folds_rest = custom_data_kfold(df_rest, target, num_splits, num_bins, random_state)

folds_all, df_all = combine_split(df_experts1, folds_experts1, df_rest, folds_rest)

print(len(df_all))
for (train, test) in folds_all:
    print(len(train), len(test))

In [ ]:
if dataset == 'dft':
    features_fix = ['vibrational_frequencies_mean', 'vibrational_frequencies_max', 'vibrational_frequencies_min']
    df_all, new_features = complex_data_conversion(df_all, features_fix)
    features_all = [f for f in features_all if f not in features_fix] + new_features
    df_all = df_all[features_all + [target, 'smiles']]
df_all.head()

In [ ]:
df_all.drop(columns=['smiles'], inplace=True)

if dataset == 'custom':
    cat_features = ['symmetry']
    num_features = [f for f in df_all.columns if f not in [target]]
else:
    cat_features = []
    num_features = [f for f in df_all.columns if f not in [target]]

df_all = prepare_data_for_regressors(df_all, num_features, cat_features)
features = [f for f in df_experts if f in features_all]
df_all.head()

## Model evaluation

In [ ]:
date = datetime.date.today().strftime('%d-%m-%Y')
save_dir = f'../../../results_{dataset}/all/{date}/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
batch_train_and_eval(df_all, 'all', folds_all, target, features, 'grid_search', save_dir)